# Single-grain coercivity results

Load all `.npz` outputs produced by `single_grain_coercivity.py` or `run_script.sh` from the local `results/` directory, collect coercivity metadata, and plot coercivity versus grain size.


In [ ]:
from pathlib import Path
import re
import math

import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")
RESULT_PATTERN = "single_grain*.npz"


def scalar(data, key, default=np.nan) -> object:
    """Load a scalar or array from an npz file."""
    if key not in data.files:
        return default
    value = data[key]
    if np.asarray(value).shape == ():
        return value.item()
    return value


def infer_run_metadata_from_filename(path: Path) -> dict:
    """
    Infer adaptive/periodic metadata from the output filename.

    Expected examples:
        single_grain_sf10_n10_P_A_FS1.0e-03.npz
        single_grain_sf10_n10_A_FS1.0e-03.npz
        single_grain_sf10_n10_P.npz
        single_grain_sf10_n10.npz
    """
    stem = path.stem

    periodic = bool(re.search(r"_n\d+_P(?:_|$)", stem))
    adaptive_match = re.search(
        r"_A_FS(?P<dh>[+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)",
        stem,
    )

    adaptive = adaptive_match is not None
    adaptive_dh_min_t = float(adaptive_match.group("dh")) if adaptive else np.nan

    return {
        "adaptive": adaptive,
        "periodic": periodic,
        "adaptive_dh_min_t": adaptive_dh_min_t,
    }


result_files = sorted(RESULTS_DIR.glob(RESULT_PATTERN))

print(f"Found {len(result_files)} result file(s) in {RESULTS_DIR.resolve()}")
for path in result_files:
    print(path)

In [ ]:
rows = []

for path in result_files:
    metadata = infer_run_metadata_from_filename(path)

    with np.load(path, allow_pickle=True) as data:
        Hc_A_per_m = float(scalar(data, "Hc_A_per_m", scalar(data, "Hc")))
        Hc_T = float(scalar(data, "Hc_T", np.nan))

        rows.append({
            "file": path.name,
            "path": path,
            "L_m": float(scalar(data, "L")),
            "size_factor": float(scalar(data, "size_factor")),
            "n": int(scalar(data, "n")),
            "ntot": int(scalar(data, "ntot")),
            "use_fmm": bool(scalar(data, "use_fmm", False)),
            "cuda": bool(scalar(data, "cuda", False)),
            "cvode": bool(scalar(data, "cvode", False)),
            "adaptive": bool(metadata["adaptive"]),
            "periodic": bool(metadata["periodic"]),
            "adaptive_dh_min_t": float(metadata["adaptive_dh_min_t"]),
            "Hc_A_per_m": Hc_A_per_m,
            "Hc_T": Hc_T,
            "abs_Hc_T": abs(Hc_T) if np.isfinite(Hc_T) else np.nan,
            "runtime_s": float(scalar(data, "runtime")),
        })

rows = sorted(
    rows,
    key=lambda row: (
        row["adaptive"],
        row["periodic"],
        np.inf if np.isnan(row["adaptive_dh_min_t"]) else row["adaptive_dh_min_t"],
        row["use_fmm"],
        row["n"],
        row["size_factor"],
    ),
)

if not rows:
    print(
        "No result files found. Run ./run_script.sh or "
        "single_grain_coercivity.py first."
    )
else:
    print(
        "file\tsize_factor\tL_m\tn\tntot\tadaptive\tperiodic\t"
        "adaptive_dh_min_t\tuse_fmm\tcuda\tHc_A_per_m\tHc_T\truntime_s"
    )

    for row in rows:
        dh_min = row["adaptive_dh_min_t"]
        dh_min_str = "nan" if np.isnan(dh_min) else f"{dh_min:.1e}"

        print(
            f"{row['file']}\t"
            f"{row['size_factor']:.6g}\t"
            f"{row['L_m']:.6e}\t"
            f"{row['n']}\t"
            f"{row['ntot']}\t"
            f"{row['adaptive']}\t"
            f"{row['periodic']}\t"
            f"{dh_min_str}\t"
            f"{row['use_fmm']}\t"
            f"{row['cuda']}\t"
            f"{row['Hc_A_per_m']:.6e}\t"
            f"{row['Hc_T']:.6e}\t"
            f"{row['runtime_s']:.3f}"
        )

    print()
    print(f"Loaded {len(rows)} result(s).")
    print(f"Unique size factors: {sorted({row['size_factor'] for row in rows})}")
    print(f"Unique resolutions:  {sorted({row['n'] for row in rows})}")

    dh_values = sorted({
        row["adaptive_dh_min_t"]
        for row in rows
        if np.isfinite(row["adaptive_dh_min_t"])
    })
    print(f"Unique dh_min values: {dh_values}")

In [ ]:
if not rows:
    print("No results to plot.")
else:
    PLOT_ABS_HC = True

    y_key = "abs_Hc_T" if PLOT_ABS_HC else "Hc_T"
    y_label = r"$|\mu_0 H_c|$ [T]" if PLOT_ABS_HC else r"$\mu_0 H_c$ [T]"

    # Keep adaptive runs with a valid dh_min.
    # Include BOTH periodic and non-periodic runs.
    plot_rows = [
        row for row in rows
        if row["adaptive"]
        and not row["use_fmm"]
        and np.isfinite(row["adaptive_dh_min_t"])
    ]

    dh_values = sorted({row["adaptive_dh_min_t"] for row in plot_rows})

    if not dh_values:
        print("No adaptive results with dh_min values found.")
    else:
        n_panels = len(dh_values)
        ncols = min(3, n_panels)
        nrows = math.ceil(n_panels / ncols)

        fig, axes = plt.subplots(
            nrows=nrows,
            ncols=ncols,
            figsize=(5.5 * ncols, 4.5 * nrows),
            squeeze=False,
            sharey=True,
        )

        axes_flat = axes.ravel()

        for ax, dh_min in zip(axes_flat, dh_values):
            rows_dh = [
                row for row in plot_rows
                if np.isclose(row["adaptive_dh_min_t"], dh_min)
            ]

            resolutions = sorted({row["n"] for row in rows_dh})

            for n in resolutions:
                for periodic in [True, False]:
                    subset = sorted(
                        [
                            row for row in rows_dh
                            if row["n"] == n and row["periodic"] == periodic
                        ],
                        key=lambda row: row["size_factor"],
                    )

                    if not subset:
                        continue

                    x = [row["size_factor"] for row in subset]
                    y = [row[y_key] for row in subset]

                    linestyle = "-" if periodic else "--"
                    label = f"n={n}, P" if periodic else f"n={n}, non-P"

                    ax.plot(x, y, ".", linestyle=linestyle, label=label)

            ax.set_title(rf"$\Delta H_\min = {dh_min:g}$ T")
            ax.set_xlabel("grain size / characteristic length [-]")
            ax.set_ylabel(y_label)
            ax.grid(True, linestyle="--", linewidth=0.5)
            ax.legend(fontsize=8)

        for ax in axes_flat[n_panels:]:
            ax.set_visible(False)

        fig.suptitle(
            "Single-grain coercivity sweep by adaptive minimum field step\n"
            "solid = periodic, dashed = non-periodic"
        )
        fig.tight_layout()